# Special-Input Determinism Diagnostic

This notebook checks whether the **paper-style special-input MSD branches**
`msd_special_x`, `msd_special_y`, and `msd_special_z` produce deterministic observables in the noiseless limit.

It looks at two frames:
- `run_detectors=False`: raw observables reconstructed from the measurement/post-processing path.
- `run_detectors=True`: detector-sampler observables, i.e. Stim-style observable flips relative to the noiseless reference sample.

The second frame is the one used by the paper-side MLD training flow.
        


In [1]:
import math
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not locate repo root containing demo/msd_utils.')

from bloqade.lanes import GeminiLogicalSimulator
from bloqade.lanes.steane_defaults import steane7_m2dets, steane7_m2obs

from demo.msd_utils.circuits import (
    build_paper_decoder_kernel_bundle,
    build_task_map,
    make_noisy_steane7_initializer,
)
from demo.msd_utils.core import run_task, split_factory_bits
from demo.msd_utils.noise import (
    DistillationSimNoiseConfig,
    build_distillation_sim_noise_model,
)
        


In [2]:
IDEAL_THETA = 0.3041 * math.pi
IDEAL_PHI = 0.25 * math.pi
IDEAL_LAM = 0.0
THETA_OFFSET = 0.30
PHI_OFFSET = 0.0
LAM_OFFSET = 0.0

THETA = IDEAL_THETA + THETA_OFFSET
PHI = IDEAL_PHI + PHI_OFFSET
LAM = IDEAL_LAM + LAM_OFFSET

OUTPUT_LOGICAL = 3
CHECKS_PER_LOGICAL = 3
SHOTS = 512

noise_model = build_distillation_sim_noise_model(
    DistillationSimNoiseConfig(
        global_scalar=1.0,
        loss_scalar=0.0,
        move_duration_us=600.0,
    )
)
sim = GeminiLogicalSimulator(noise_model=noise_model)
noisy_steane7_initialize = make_noisy_steane7_initializer(sim)

kernel_bundle = build_paper_decoder_kernel_bundle(
    THETA,
    PHI,
    LAM,
    output_qubit=OUTPUT_LOGICAL,
)

special_tasks = build_task_map(
    sim,
    kernel_bundle.special,
    m2dets=steane7_m2dets(5),
    m2obs=steane7_m2obs(5),
    noisy_initializer=noisy_steane7_initialize,
    append_measurements=False,
)

list(special_tasks)
        


('callee must be a method type, got function', 'when verifying the following statement', ' `func.Call` at\n', '    \x1bfunc\x1b\x1b.\x1b\x1bcall\x1b %0(%1, %2)\x1b : \x1b\x1b!\x1b\x1bBottom\x1b\x1b maybe_pure=False\x1b\n')


In [ ]:
def summarize_unique_rows(array: np.ndarray):
    unique, counts = np.unique(array.astype(np.uint8), axis=0, return_counts=True)
    return list(zip(unique.tolist(), counts.tolist()))


def summarize_special_task(task, *, shots: int):
    raw = run_task(task, shots, with_noise=False, run_detectors=False)
    det = run_task(task, shots, with_noise=False, run_detectors=True)

    raw_anc_det, raw_anc_obs = split_factory_bits(
        raw.detectors,
        raw.observables,
        output_logical=OUTPUT_LOGICAL,
        checks_per_logical=CHECKS_PER_LOGICAL,
    )
    det_anc_det, det_anc_obs = split_factory_bits(
        det.detectors,
        det.observables,
        output_logical=OUTPUT_LOGICAL,
        checks_per_logical=CHECKS_PER_LOGICAL,
    )

    return {
        'raw_full_observables': summarize_unique_rows(raw.observables),
        'raw_output_observable': np.unique(raw.observables[:, OUTPUT_LOGICAL].astype(np.uint8)).tolist(),
        'raw_ancilla_observables': summarize_unique_rows(raw_anc_obs),
        'raw_detectors': summarize_unique_rows(raw.detectors),
        'detector_sampler_full_observables': summarize_unique_rows(det.observables),
        'detector_sampler_output_observable': np.unique(det.observables[:, OUTPUT_LOGICAL].astype(np.uint8)).tolist(),
        'detector_sampler_ancilla_observables': summarize_unique_rows(det_anc_obs),
        'detector_sampler_detectors': summarize_unique_rows(det.detectors),
        'detector_sampler_zero_obs_all_shots': bool(np.all(det.observables == 0)),
        'detector_sampler_zero_det_all_shots': bool(np.all(det.detectors == 0)),
        'raw_shape': (raw.detectors.shape, raw.observables.shape),
        'detector_shape': (det.detectors.shape, det.observables.shape),
    }
        


In [3]:
results = {
    basis: summarize_special_task(task, shots=SHOTS)
    for basis, task in special_tasks.items()
}

results
        


NameError: name 'special_tasks' is not defined

In [ ]:
for basis in 'XYZ':
    summary = results[basis]
    print(f'===== {basis} =====')
    print('raw full observables:', summary['raw_full_observables'])
    print('raw output logical unique:', summary['raw_output_observable'])
    print('raw ancilla observables:', summary['raw_ancilla_observables'])
    print('raw detector patterns:', summary['raw_detectors'])
    print('detector-sampler full observables:', summary['detector_sampler_full_observables'])
    print('detector-sampler output logical unique:', summary['detector_sampler_output_observable'])
    print('detector-sampler ancilla observables:', summary['detector_sampler_ancilla_observables'])
    print('detector-sampler detector patterns:', summary['detector_sampler_detectors'])
    print('detector-sampler all-zero observables:', summary['detector_sampler_zero_obs_all_shots'])
    print('detector-sampler all-zero detectors:', summary['detector_sampler_zero_det_all_shots'])
    print()
        


## How To Read This

If the paper-style training condition is reproduced, the strongest expected signal is:
- in the **detector-sampler frame**, noiseless observables should collapse to a single pattern, ideally all zeros,
- and noiseless detectors should also collapse to a single pattern, ideally all zeros.

The raw-measurement frame is stricter. If it is not deterministic there, that does **not** by itself mean the MLD training frame is wrong.
        
